## Fitting Min 7-Day Average Flow Data from Russian River Basins to a Pearson-III Distribution Using the Method of Moments to Quantify Extremeness of Years
by Arman Omidvar

Analysis of Low Flows and Selected Methods for Estimating Low-Flow Characteristics at Partial-Record and Ungaged Stream Sites in Western Washington

In [ ]:
import sys
sys.path.insert(0, '../..')

import numpy as np
import pandas as pd
from scipy.stats import pearson3
import matplotlib.pyplot as plt
from scipy.special import gammaln
from UCB_training.UCB_utils import clean_df

In [ ]:
def sample_mean(low_flows):
    log_low_flows = np.log10(low_flows)
    return np.mean(log_low_flows)

def sample_std(low_flows):
    log_low_flows = np.log10(low_flows)
    return np.std(log_low_flows, ddof=1)

def sample_skew(low_flows):
    n = len(low_flows)
    log_low_flows = np.log10(low_flows)
    std = np.std(log_low_flows, ddof=1)
    mean = np.mean(log_low_flows)

    return (np.sum((log_low_flows - mean) ** 3) * n) / ((n - 1) * (n - 2) * (std ** 3))

In [ ]:
def pearsonIII_alpha_estimate(sample_skew):
    """
    Compute the MOM estimate of the alpha parameter for a Pearson-III distribution based on the weighted skew.
    """
    alpha = (4 / (sample_skew ** 2))
    return alpha

def pearsonIII_beta_estimate(sample_skew, standard_error, alpha_hat):
    """
    Compute the MOM estimate of the beta parameter for a Pearson-III distribution.
    """
    beta_hat = np.sign(sample_skew) * (standard_error ** 2 / alpha_hat) ** 0.5
    return beta_hat

def pearsonIII_tau_estimate(mean_log_peak_flows, beta_hat, alpha_hat):
    """
    Compute the MOM estimate of the tau parameter for a Pearson-III distribution.
    """
    tau_hat = mean_log_peak_flows - (alpha_hat * beta_hat)
    return tau_hat

In [ ]:
calpella_low_flows = pd.read_csv('extreme_year_analysis/data/low_flows_7d_avg/Capella.csv')[1:][:-1]
guerneville_low_flows = pd.read_csv('extreme_year_analysis/data/low_flows_7d_avg/Guerneville.csv')[1:][:-1]
hopland_low_flows = pd.read_csv('extreme_year_analysis/data/low_flows_7d_avg/Hopland.csv')[1:][:-1]
warmsprings_low_flows = pd.read_csv('extreme_year_analysis/data/low_flows_7d_avg/Warm.csv')[1:][:-1]

calpella_all_data = clean_df(pd.read_csv('../../russian_river_data/Calpella_daily.csv')).reset_index().rename(columns={'date': 'Date'})
guerneville_all_data = clean_df(pd.read_csv('../../russian_river_data/Guerneville_daily.csv')).reset_index().rename(columns={'date': 'Date'})
hopland_all_data = clean_df(pd.read_csv('../../russian_river_data/Hopland_daily.csv')).reset_index().rename(columns={'date': 'Date'})
warmsprings_all_data = clean_df(pd.read_csv('../../russian_river_data/WarmSprings_Inflow_daily.csv')).reset_index().rename(columns={'date': 'Date'})

cutoff = pd.Timestamp('2009-09-30')
calpella_all_data = calpella_all_data[calpella_all_data['Date'] <= cutoff]
guerneville_all_data = guerneville_all_data[guerneville_all_data['Date'] <= cutoff]
hopland_all_data = hopland_all_data[hopland_all_data['Date'] <= cutoff]
warmsprings_all_data = warmsprings_all_data[warmsprings_all_data['Date'] <= cutoff]

low_flows = {'Calpella': calpella_low_flows['mav'].values,
              'Guerneville': guerneville_low_flows['mav'].values,
              'Hopland': hopland_low_flows['mav'].values,
              'Warm Springs': warmsprings_low_flows['mav'].values}

all_data = {'Calpella': calpella_all_data,
            'Guerneville': guerneville_all_data,
            'Hopland': hopland_all_data,
            'Warm Springs': warmsprings_all_data}


extreme_years = pd.DataFrame()
extreme_years['climatic_year'] = calpella_low_flows['climatic_year']


def pearson3_pdf(x, alpha, beta, tau):
    x = np.asarray(x, dtype=float)

    if alpha <= 0 or beta == 0:
        return np.full_like(x, np.nan)

    z = (x - tau) / beta
    mask = z > 0

    out = np.zeros_like(x, dtype=float)
    logpdf = -np.log(abs(beta)) - gammaln(alpha) + (alpha - 1)*np.log(z[mask]) - z[mask]
    out[mask] = np.exp(logpdf)
    return out

for basin in low_flows.keys():
    basin_low_flows = low_flows[basin]
    sam_skew = sample_skew(basin_low_flows)

    log_flows = np.log10(basin_low_flows)
    mean_log_flows = np.mean(log_flows)
    std_log_flows = np.std(log_flows, ddof=1)

    alpha_hat = pearsonIII_alpha_estimate(sam_skew)
    beta_hat = pearsonIII_beta_estimate(sam_skew, std_log_flows, alpha_hat)
    tau_hat = pearsonIII_tau_estimate(mean_log_flows, beta_hat, alpha_hat)

    print(f'Basin: {basin}')
    print(f'alpha_hat: {alpha_hat}, beta_hat: {beta_hat}, tau_hat: {tau_hat}')

    dist = pearson3(sam_skew, loc=mean_log_flows, scale=std_log_flows)
    x = np.linspace(min(log_flows), max(log_flows), 200)
    scipy_pdf_vals = dist.pdf(x)
    my_pdf_vals = pearson3_pdf(x, alpha_hat, beta_hat, tau_hat)

    extreme_years[basin] = dist.cdf(np.log10(basin_low_flows))

    plt.figure(figsize=(10, 6))
    plt.hist(log_flows, bins=15, density=True, alpha=0.5, label='Observed log-flows')
    plt.plot(x, my_pdf_vals, 'b-', lw=2, label='Pearson III fit')
    plt.plot(x, scipy_pdf_vals, 'r--', lw=2, label='SciPy Pearson III fit')
    plt.xlabel('log10(Q)')
    plt.ylabel('Density')
    plt.title(f'Distribution of log low flows for {basin}')
    plt.legend()
    plt.show()

    flows = all_data[basin].copy()
    flows['climatic_year'] = np.where(flows['Date'].dt.month >= 4, flows['Date'].dt.year + 1, flows['Date'].dt.year)
    flows['mav'] = None
    flows['extremeness'] = None
    for i, year in enumerate(flows['climatic_year'].unique()[1:][:-1]):
        flows.loc[flows['climatic_year'] == year, 'mav'] = basin_low_flows[i]
        flows.loc[flows['climatic_year'] == year, 'extremeness'] = extreme_years[basin].iloc[i]


    plt.bar(
        flows['Date'],
        flows['extremeness'] * max(all_data[basin].iloc[:, 1]),
        color="blue",
        alpha=0.3,
        label="Extremeness"
    )

    plt.plot(
        all_data[basin]['Date'],
        all_data[basin].iloc[:, 1],
        color="black",
        linewidth=0.5,
        label="Daily Flow",
    )

    plt.legend()
    plt.figure(figsize=(300, 100))
    plt.show()

In [ ]:
extreme_years.to_csv('extreme_year_analysis/data/low_flow_analysis.csv', index=False)
print(extreme_years)

In [ ]:
basins = [col for col in extreme_years.columns if col != "climatic_year"]

for basin in basins:
    print(f"\n=== {basin} (lowest → highest extremeness) ===")
    
    ranked = (extreme_years[["climatic_year", basin]].sort_values(by=basin, ascending=True)  # lowest = most extreme
        .reset_index(drop=True))
    
    ranked["rank"] = ranked.index + 1
    
    print(ranked[["rank", "climatic_year", basin]])